# Extração Completa de AUs - Dataset BAH
Py-Feat com GPU | Batch processing | Resume automático

In [2]:
!pip install py-feat

In [5]:
!pip install "scipy<1.14"

   ---------------------------------------- 0.0/46.2 MB ? eta -:--:--
    --------------------------------------- 0.8/46.2 MB 6.7 MB/s eta 0:00:07
   ------- -------------------------------- 8.4/46.2 MB 27.4 MB/s eta 0:00:02
   ---------- ----------------------------- 12.3/46.2 MB 24.9 MB/s eta 0:00:02
   ------------------- -------------------- 22.0/46.2 MB 31.6 MB/s eta 0:00:01
   -------------------------- ------------- 30.7/46.2 MB 34.8 MB/s eta 0:00:01
   --------------------------------- ------ 38.8/46.2 MB 34.3 MB/s eta 0:00:01
   ---------------------------------------  45.6/46.2 MB 34.1 MB/s eta 0:00:01
   ---------------------------------------- 46.2/46.2 MB 32.0 MB/s  0:00:01
  Attempting uninstall: scipy
    Found existing installation: scipy 1.15.3
    Uninstalling scipy-1.15.3:
      Successfully uninstalled scipy-1.15.3


  You can safely remove it manually.
  You can safely remove it manually.


In [1]:
# CÉLULA 1: Imports e configuração
import os
import re
import time
import numpy as np
from tqdm.notebook import tqdm
from feat import Detector

# === AJUSTE ESSES PATHS ===
#DIRECTORY_VARIABLES NEED TO CHANGE TO YOUR CORRECT DIRECTORIES
BASE_DIR =  r'\ABAW\data'

FACES_DIR = os.path.join(BASE_DIR, "cropped-aligned-faces", "Videos")
OUTPUT_DIR = os.path.join(BASE_DIR, "au_features")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Ajuste conforme VRAM: 16 pra 4GB, 32 pra 8GB, 64 pra 16GB+
BATCH_SIZE = 32

AU_COLS = [
    'AU01', 'AU02', 'AU04', 'AU05', 'AU06', 'AU07', 'AU09', 'AU10',
    'AU11', 'AU12', 'AU14', 'AU15', 'AU17', 'AU20', 'AU23', 'AU24',
    'AU25', 'AU26', 'AU28', 'AU43'
]

print(f"Faces dir: {FACES_DIR}")
print(f"Output dir: {OUTPUT_DIR}")

Faces dir: C:\Users\ddonz\OneDrive\Documentos\Aislan\data\cropped-aligned-faces\Videos
Output dir: C:\Users\ddonz\OneDrive\Documentos\Aislan\data\au_features


In [2]:
# CÉLULA 2: Inicializar detector com GPU
detector = Detector(device="cuda")
print("Detector pronto com GPU!")

Detector pronto com GPU!


In [3]:
# CÉLULA 3: Descobrir todos os vídeos e checar resume
def frame_sort_key(filename):
    match = re.search(r'frame-(\d+)', filename)
    return int(match.group(1)) if match else 0

all_videos = []
participants = sorted(os.listdir(FACES_DIR))

for pid in participants:
    visit_path = os.path.join(FACES_DIR, pid, "Visite_1")
    if not os.path.isdir(visit_path):
        continue
    for video_name in sorted(os.listdir(visit_path)):
        v_path = os.path.join(visit_path, video_name)
        if os.path.isdir(v_path):
            all_videos.append((pid, video_name, v_path))

# Checar quais já foram feitos
to_process = []
already_done = 0
for pid, video_name, v_path in all_videos:
    out_path = os.path.join(OUTPUT_DIR, pid, f"{video_name}.npy")
    if os.path.exists(out_path):
        already_done += 1
    else:
        to_process.append((pid, video_name, v_path))

print(f"Total de vídeos: {len(all_videos)}")
print(f"Já processados: {already_done}")
print(f"A processar: {len(to_process)}")

Total de vídeos: 1427
Já processados: 1187
A processar: 240


In [4]:
# CÉLULA DE DIAGNÓSTICO - rodar antes de continuar
import torch
print(f"CUDA disponível: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Detector device: {detector.device}")

# Testar 1 frame pra ver tempo
import time
test_pid = os.listdir(FACES_DIR)[0]
test_visit = os.path.join(FACES_DIR, test_pid, "Visite_1")
test_video = os.listdir(test_visit)[0]
test_path = os.path.join(test_visit, test_video)
test_frames = sorted([os.path.join(test_path, f) for f in os.listdir(test_path) if f.endswith('.jpg')])[:10]

# Teste 1: um frame
t0 = time.time()
r = detector.detect_image(test_frames[0])
print(f"\n1 frame: {time.time()-t0:.2f}s")

# Teste 2: batch de 10
t0 = time.time()
r = detector.detect_image(test_frames)
print(f"10 frames (batch): {time.time()-t0:.2f}s")

CUDA disponível: True
GPU: NVIDIA GeForce RTX 5060 Ti
Detector device: cuda


100%|██████████| 1/1 [00:01<00:00,  1.81s/it]



1 frame: 2.19s


100%|██████████| 10/10 [00:06<00:00,  1.60it/s]

10 frames (batch): 6.42s


In [5]:
# CÉLULA 4: Extração principal — amostragem ~10fps + otimizações

def extract_aus_batch(detector, frame_paths, batch_size=32):
    """Extrai AUs em batch. Retorna (N_frames, 20) float64."""
    all_aus = []
    n_failed = 0

    for i in range(0, len(frame_paths), batch_size):
        batch_paths = frame_paths[i:i + batch_size]
        try:
            results = detector.detect_image(batch_paths)
            aus_batch = results[AU_COLS].values.astype(np.float64)

            if len(aus_batch) < len(batch_paths):
                padding = np.zeros((len(batch_paths) - len(aus_batch), 20))
                aus_batch = np.vstack([aus_batch, padding])
                n_failed += len(batch_paths) - len(aus_batch)

            all_aus.append(aus_batch)
        except Exception as e:
            for fp in batch_paths:
                try:
                    result = detector.detect_image(fp)
                    aus = result[AU_COLS].values[0].astype(np.float64)
                    all_aus.append(aus.reshape(1, -1))
                except:
                    all_aus.append(np.zeros((1, 20)))
                    n_failed += 1

    if len(all_aus) == 0:
        return np.zeros((0, 20)), 0
    return np.vstack(all_aus), n_failed


# --- Loop principal ---
SAMPLE_RATE = 3  # 1 a cada 3 frames (~30fps → ~10fps)

errors = []
total_frames = 0
start_time = time.time()

for idx, (pid, video_name, v_path) in enumerate(tqdm(to_process, desc="Vídeos")):
    pid_out_dir = os.path.join(OUTPUT_DIR, pid)
    os.makedirs(pid_out_dir, exist_ok=True)
    out_path = os.path.join(pid_out_dir, f"{video_name}.npy")

    # Listar, ordenar e AMOSTRAR frames
    frames = [f for f in os.listdir(v_path) if f.endswith(('.jpg', '.png'))]
    frames = sorted(frames, key=frame_sort_key)[::SAMPLE_RATE]
    frame_paths = [os.path.join(v_path, f) for f in frames]

    if len(frame_paths) == 0:
        errors.append(f"{pid}/{video_name}: sem frames")
        continue

    try:
        aus_array, n_failed = extract_aus_batch(detector, frame_paths, BATCH_SIZE)
        np.save(out_path, aus_array)
        total_frames += len(frame_paths)

        if n_failed > 0:
            errors.append(f"{pid}/{video_name}: {n_failed}/{len(frame_paths)} frames falharam")

    except Exception as e:
        errors.append(f"{pid}/{video_name}: {str(e)}")

    # Print progresso a cada 100 vídeos
    if (idx + 1) % 100 == 0:
        elapsed = time.time() - start_time
        fps = total_frames / elapsed if elapsed > 0 else 0
        eta_h = (sum(
            len([f for f in os.listdir(v) if f.endswith(('.jpg', '.png'))]) // SAMPLE_RATE
            for _, _, v in to_process[idx+1:]
        ) / fps / 3600) if fps > 0 else 0
        print(f"  {idx+1}/{len(to_process)} | {total_frames} frames | "
              f"{fps:.1f} f/s | ETA ~{eta_h:.1f}h")

elapsed = time.time() - start_time
print(f"\nConcluído! {len(to_process)} vídeos | {total_frames} frames | {elapsed/60:.1f} min")
print(f"Velocidade: {total_frames/elapsed:.1f} frames/s")
print(f"Amostragem: 1/{SAMPLE_RATE} (~10fps)")
if errors:
    print(f"\n{len(errors)} erros encontrados")


100%|██████████| 32/32 [00:22<00:00,  1.45it/s]

100%|██████████| 32/32 [00:21<00:00,  1.47it/s]

100%|██████████| 32/32 [00:21<00:00,  1.46it/s]

100%|██████████| 24/24 [00:15<00:00,  1.55it/s]

100%|██████████| 32/32 [00:21<00:00,  1.50it/s]

100%|██████████| 32/32 [00:21<00:00,  1.48it/s]

100%|██████████| 32/32 [00:21<00:00,  1.51it/s]

100%|██████████| 32/32 [00:20<00:00,  1.55it/s]

100%|██████████| 11/11 [00:07<00:00,  1.51it/s]

100%|██████████| 32/32 [00:21<00:00,  1.52it/s]

100%|██████████| 32/32 [00:21<00:00,  1.48it/s]

100%|██████████| 32/32 [00:21<00:00,  1.51it/s]

100%|██████████| 32/32 [00:20<00:00,  1.55it/s]

100%|██████████| 32/32 [00:21<00:00,  1.51it/s]

100%|██████████| 4/4 [00:02<00:00,  1.48it/s]

100%|██████████| 32/32 [00:21<00:00,  1.48it/s]

100%|██████████| 32/32 [00:21<00:00,  1.51it/s]

100%|██████████| 32/32 [00:20<00:00,  1.56it/s]

100%|██████████| 32/32 [00:21<00:00,  1.51it/s]

100%|██████████| 32/32 [00:21<00:00,  1.49it/s]

100%|██████████| 32/3

  200/240 | 39507 frames | 1.2 f/s | ETA ~2.2h



100%|██████████| 32/32 [00:21<00:00,  1.47it/s]

100%|██████████| 32/32 [00:21<00:00,  1.46it/s]

100%|██████████| 32/32 [00:20<00:00,  1.53it/s]

100%|██████████| 32/32 [00:22<00:00,  1.45it/s]

100%|██████████| 32/32 [00:21<00:00,  1.46it/s]

100%|██████████| 2/2 [00:01<00:00,  1.62it/s]

100%|██████████| 32/32 [00:21<00:00,  1.46it/s]

100%|██████████| 32/32 [00:22<00:00,  1.44it/s]

100%|██████████| 32/32 [00:21<00:00,  1.49it/s]

100%|██████████| 32/32 [00:21<00:00,  1.50it/s]

100%|██████████| 32/32 [00:21<00:00,  1.50it/s]

100%|██████████| 32/32 [00:21<00:00,  1.49it/s]

100%|██████████| 32/32 [00:21<00:00,  1.47it/s]

100%|██████████| 32/32 [00:21<00:00,  1.50it/s]

100%|██████████| 32/32 [00:20<00:00,  1.53it/s]

100%|██████████| 26/26 [00:18<00:00,  1.44it/s]

100%|██████████| 32/32 [00:22<00:00,  1.45it/s]

100%|██████████| 32/32 [00:20<00:00,  1.53it/s]

100%|██████████| 32/32 [00:21<00:00,  1.50it/s]

100%|██████████| 32/32 [00:21<00:00,  1.48it/s]

100%|██████████| 32/3


Concluído! 240 vídeos | 48582 frames | 677.7 min
Velocidade: 1.2 frames/s
Amostragem: 1/3 (~10fps)


In [6]:
# CÉLULA 5: Verificar erros (se houver)
if errors:
    print(f"Total de erros: {len(errors)}\n")
    for e in errors[:20]:  # mostrar primeiros 20
        print(f"  - {e}")
    if len(errors) > 20:
        print(f"  ... e mais {len(errors)-20}")
else:
    print("Nenhum erro!")

Nenhum erro!


In [7]:
# CÉLULA 6: Validação - verificar o que foi salvo
saved_files = []
for pid in os.listdir(OUTPUT_DIR):
    pid_dir = os.path.join(OUTPUT_DIR, pid)
    if not os.path.isdir(pid_dir):
        continue
    for f in os.listdir(pid_dir):
        if f.endswith('.npy'):
            fpath = os.path.join(pid_dir, f)
            arr = np.load(fpath)
            saved_files.append((pid, f, arr.shape))

print(f"Arquivos .npy salvos: {len(saved_files)}")
print(f"Participantes: {len(set(p for p, _, _ in saved_files))}")

# Estatísticas de frames
frame_counts = [s[0] for _, _, s in saved_files]
print(f"Frames por vídeo: min={min(frame_counts)}, max={max(frame_counts)}, "
      f"média={np.mean(frame_counts):.0f}, total={sum(frame_counts)}")

# Checar se todos os 20 AUs estão presentes
au_dims = set(s[1] for _, _, s in saved_files)
print(f"Dimensão AU: {au_dims}")

Arquivos .npy salvos: 1427
Participantes: 300
Frames por vídeo: min=25, max=765, média=215, total=306156
Dimensão AU: {20}
